# Amazon Fine Food Reviews: fine-tuning a transformer on product reviews

**Task:** classify a product review as `negative`, `neutral` or `positive`, using the star rating as the label (1-2 stars = negative, 3 = neutral, 4-5 = positive).

**Data:** Amazon Fine Food Reviews on Kaggle (`Reviews.csv`, ~568k reviews): <https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews>. Any review CSV with a text column and a 1-5 rating works (for example *Women's E-Commerce Clothing Reviews*); the column names are detected automatically, or set them in the configuration cell.

**Central question:** *How much does fine-tuning a transformer improve over TF-IDF + Logistic Regression on long, noisy product reviews, and does the result hold on the natural (imbalanced) rating distribution and on a different domain (tweets)?*

## Design decisions specific to reviews

| Issue | What the notebook does |
|---|---|
| Huge, very imbalanced dataset (about 78 % positive, 14 % negative, 8 % neutral) | A **capped per-class sample** (default 10,000 per class) for training, so the model must learn all three classes |
| A balanced test set hides how the model behaves on real data | An extra **natural-distribution check** on a random holdout (about 5,000 reviews) that is set aside before sampling |
| Duplicate review texts across products inflate scores | Exact duplicates are removed **before** splitting |
| HTML fragments such as `<br />` in the text | Removed and HTML entities decoded |
| Star ratings are only a proxy for sentiment (a 3-star review can be mixed or mildly positive) | Stated as a limitation; neutral is expected to be the hardest class |

## What this notebook covers

| # | Section | Why it matters |
|---|---|---|
| 1-2 | Loading, label mapping, sampling, EDA | understand the data before modelling |
| 3-4 | Cleaning and a stratified train / validation / test split | no leakage, test set used once |
| 5-7 | Baselines, stopword ablation, tuned Logistic Regression | every complex model has to beat a simple one |
| 8 | Fine-tuned transformer (RoBERTa-base) | the main experiment |
| 9-10 | Model comparison with speed, final test evaluation | honest final numbers |
| 11 | Natural-distribution check and cross-domain check (tweets) | how does it behave on realistic and unfamiliar data? |
| 12-13 | Interpretability and error analysis | what did the models learn, where do they fail? |
| 14 | Saving the models for the Streamlit app | the deployable output |

**Metrics:** macro-F1 (every class counts equally) next to accuracy, precision and recall.

## 0. Setup and configuration

In [ ]:
SEED = 42

DATA_PATH = None                 # path to your reviews CSV; None = search the usual Kaggle / local locations
TEXT_COL = None                  # None = auto-detect (e.g. "Text", "Review Text")
RATING_COL = None                # None = auto-detect (e.g. "Score", "Rating")
USE_SUMMARY = False              # True = prepend the review title/summary (column "Summary" or "Review Title") to the text
PER_CLASS_CAP = 10000            # max reviews per class used for train/val/test (None = use everything; slow)
NATURAL_CHECK_N = 5000           # random holdout with the natural class mix (capped at 10% of the data), used only for a final check

#transformer
RUN_TRANSFORMER = True          
USE_ALL_GPUS = True           
TRANSFORMER_NAME = "roberta-base"  
TRANSFORMER_EPOCHS = 10        
TRANSFORMER_BATCH = 32
TRANSFORMER_LR = 3e-5
MAX_LEN = 128                    
TRANSFORMER_TRAIN_SIZE = None    

#cross-domain check
CROSS_DATASET_CSV = None         
CROSS_DATASET_N = 3000       

MODELS_DIR = "models"           
ASSETS_DIR = "assets"           

In [ ]:
import html
import json
import platform
import random
import re
import time
from datetime import date
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 250)

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
MODELS_DIR, ASSETS_DIR = ROOT / MODELS_DIR, ROOT / ASSETS_DIR
MODELS_DIR.mkdir(exist_ok=True)
ASSETS_DIR.mkdir(exist_ok=True)

def save_fig(fig, name):
    fig.savefig(ASSETS_DIR / name, dpi=150, bbox_inches="tight")

## 1. Load the data, map ratings to sentiment, and sample

Steps: find and read the CSV, detect the text and rating columns, map stars to three classes, clean HTML, **remove duplicate review texts**, then draw a capped per-class sample. A random holdout with the **natural** class mix is set aside first (for the check in section 10b); the capped sample is drawn from the remaining reviews, so the two never overlap.

In [ ]:
data_path = "/kaggle/input/datasets/snap/amazon-fine-food-reviews/Reviews.csv"

if data_path is None:
    raise FileNotFoundError("Reviews CSV not found.")

raw = pd.read_csv(data_path)
print("Loaded:", data_path, raw.shape)
print("Columns:", list(raw.columns))

def pick(preferred, candidates, what):
    if preferred:
        assert preferred in raw.columns, f"{what} column '{preferred}' not in {list(raw.columns)}"
        return preferred
    found = next((c for c in candidates if c in raw.columns), None)
    if found is None:
        raise KeyError(f"Could not detect the {what} column; set it in the configuration cell. Columns: {list(raw.columns)}")
    return found

text_col = pick(TEXT_COL, ["Text", "Review Text", "reviewText", "review_text", "review", "text", "content"], "text")
rating_col = pick(RATING_COL, ["Score", "Rating", "overall", "rating", "stars", "star_rating", "score"], "rating")
title_col = next((c for c in ["Summary", "Review Title", "summary", "title"] if c in raw.columns), None)
print(f"Using text column '{text_col}', rating column '{rating_col}'" + (f", title column '{title_col}'" if USE_SUMMARY and title_col else ""))
raw.head()

In [ ]:
LABELS = ["negative", "neutral", "positive"]
STAR_TO_LABEL = {1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive"}

def clean_html(s):
    s = html.unescape(str(s))
    s = re.sub(r"<[^>]+>", " ", s)          # <br />, <a href=...> and similar
    return " ".join(s.split())

df_all = pd.DataFrame({"text": raw[text_col], "rating": pd.to_numeric(raw[rating_col], errors="coerce")})
if USE_SUMMARY and title_col:
    df_all["text"] = raw[title_col].fillna("").astype(str) + ". " + df_all["text"].fillna("").astype(str)

print("Missing values:\n", df_all.isna().sum().to_string(), "\n")
df_all = df_all.dropna()
df_all = df_all[df_all["rating"].isin(STAR_TO_LABEL)]
df_all["sentiment"] = df_all["rating"].astype(int).map(STAR_TO_LABEL)
df_all["text"] = df_all["text"].apply(clean_html)
df_all = df_all[df_all["text"].str.len() > 0]

n_before = len(df_all)
df_all = df_all.drop_duplicates(subset="text").reset_index(drop=True)      # BEFORE splitting: same text must not be in train and test
print(f"Removed {n_before - len(df_all):,} duplicate review texts; {len(df_all):,} unique reviews remain")

print("\nNatural class distribution (all unique reviews):")
print(df_all["sentiment"].value_counts(normalize=True).reindex(LABELS).round(3).to_string())

# 1) hold out a random sample with the NATURAL class mix first (never used for training or model selection)
n_nat = min(NATURAL_CHECK_N, int(0.10 * len(df_all)))
natural_df = df_all.sample(n_nat, random_state=SEED)
pool = df_all.drop(index=natural_df.index)
natural_df = natural_df.reset_index(drop=True)

# 2) capped per-class sample from the remaining reviews, used for train / validation / test
parts = []
for lab in LABELS:
    grp = pool[pool["sentiment"] == lab]
    n = len(grp) if PER_CLASS_CAP is None else min(len(grp), PER_CLASS_CAP)
    parts.append(grp.sample(n, random_state=SEED))
df = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\nSampled dataset used for train / val / test:", df.shape)
print(df["sentiment"].value_counts(normalize=True).reindex(LABELS).round(3).to_string())
print("Natural-distribution check set:", natural_df.shape, natural_df["sentiment"].value_counts(normalize=True).reindex(LABELS).round(3).to_dict())

## 2. Exploratory data analysis

In [ ]:
LABELS = ["negative", "neutral", "positive"]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
nat = df_all["sentiment"].value_counts().reindex(LABELS)
axes[0].bar(LABELS, nat.values); axes[0].set_title("All unique reviews (natural mix)")
smp = df["sentiment"].value_counts().reindex(LABELS)
axes[1].bar(LABELS, smp.values); axes[1].set_title("Sampled data (train / val / test)")
star = df_all["rating"].astype(int).value_counts().sort_index()
axes[2].bar(star.index.astype(str), star.values); axes[2].set_title("Star ratings (natural)"); axes[2].set_xlabel("stars")
plt.tight_layout()
save_fig(fig, "class_distribution.png")
plt.show()

In the natural data the positive class dominates, so a model that always predicts "positive" would reach about 78 % accuracy while being useless for neutral and negative reviews. The sample is capped per class so that the model has to learn all three. Because the sampled test set is balanced, its accuracy is **not** inflated by the majority class, and the natural-distribution check in section 11 shows how the model behaves on the real mix.

In [ ]:
df["n_words"] = df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.boxplot(x="sentiment", y="n_words", data=df, order=LABELS, ax=axes[0], showfliers=False)
axes[0].set_title("Words per review by sentiment")
sns.histplot(df["n_words"].clip(upper=400), bins=40, ax=axes[1])
axes[1].set_title("Review length (words, clipped at 400)")
plt.show()

p95 = int(df["n_words"].quantile(0.95))
words_fit = int(MAX_LEN * 0.75)             # rule of thumb: 1 token is about 0.75 English words
print(f"Median review: {int(df['n_words'].median())} words, 95th percentile: {p95} words")
print(f"MAX_LEN = {MAX_LEN} tokens is about {words_fit} words; {(df['n_words'] > words_fit).mean():.1%} of reviews are longer and get truncated.")
df.groupby("sentiment")["n_words"].mean().round(1).to_frame("avg words")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, label in zip(axes, LABELS):
    cv = CountVectorizer(stop_words="english", token_pattern=r"[a-zA-Z]{3,}")
    mat = cv.fit_transform(df.loc[df["sentiment"] == label, "text"])
    freq = pd.Series(np.asarray(mat.sum(axis=0)).ravel(), index=cv.get_feature_names_out())
    freq.nlargest(15).sort_values().plot.barh(ax=ax)
    ax.set_title(f"Most frequent words: {label}")
plt.tight_layout()
plt.show()

## 3. Text cleaning

The cleaning code lives in a small module, `src/text_utils.py`, which the next cell writes into the project folder. **The Streamlit app imports the same module**, so preprocessing at prediction time is identical to training time.

Two cleaning modes:

- `clean_text`: for the bag-of-words model. Lowercases, removes URLs, expands contractions so negation becomes the token `not`, shortens elongated words (`soooo` to `soo`), keeps letters only. Optional stopword removal with a **custom list that keeps negations and intensifiers** (*not, no, never, very, too...*), because those words carry sentiment ("not good", "too salty").
- `light_clean`: for the transformer. Only replaces URLs with `http` and collapses spaces; transformers are trained on raw text and do better with minimal cleaning.

HTML fragments were already removed when the data was loaded (section 1).

In [ ]:
# Write the shared cleaning module to <project>/src/text_utils.py and make it importable.
# It lives inside the notebook so that the notebook also works on its own (for example on Kaggle).
import sys

TEXT_UTILS_SOURCE = r'''
"""Shared text utilities for the Twitter sentiment project.

Used by the training notebook and by the Streamlit app, so that exactly the same
preprocessing is applied at training time and at prediction time.
"""
import re

import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

LABELS = ["negative", "neutral", "positive"]

# Words that matter for sentiment and must NOT be removed as "stopwords".
KEEP_WORDS = {"no", "not", "nor", "never", "nothing", "cannot", "without",
              "very", "too", "more", "most", "least", "against"}
STOPWORDS = set(ENGLISH_STOP_WORDS) - KEEP_WORDS

URL_RE = re.compile(r"http\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
REPEAT_RE = re.compile(r"(.)\1{2,}")


def decontract(text):
    """Expand contractions so that negation survives as the token 'not'."""
    text = text.replace("\u2019", "'")
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"can't", "can not", text)
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'s", " is", text)
    text = re.sub(r"'d", " would", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'m", " am", text)
    return text


def clean_text(text, remove_stopwords=False):
    """Heavy cleaning for bag-of-words models (TF-IDF + linear model)."""
    text = str(text).lower()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = text.replace("#", " ")
    text = decontract(text)
    text = REPEAT_RE.sub(r"\1\1", text)          # sooooo -> soo
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]
    return " ".join(tokens)


def light_clean(text):
    """Minimal cleaning for transformer models (they handle raw text well)."""
    text = URL_RE.sub("http", str(text))
    text = MENTION_RE.sub("@user", text)
    return " ".join(text.split())


def explain_linear(pipeline, cleaned_text, top_k=8):
    """Explain one prediction of a TF-IDF + linear-classifier sklearn Pipeline.

    Returns (classes, probabilities, predicted_label, contributions) where
    `contributions` is a list of (term, score) sorted by absolute size. A score is
    tfidf_value * coefficient for the predicted class: positive scores pushed the
    prediction towards that class, negative scores pushed against it.
    """
    vec = pipeline.named_steps["tfidf"]
    clf = pipeline.named_steps["clf"]
    probs = pipeline.predict_proba([cleaned_text])[0]
    classes = list(clf.classes_)
    idx = int(np.argmax(probs))

    row = vec.transform([cleaned_text]).tocoo()
    names = vec.get_feature_names_out()
    coefs = clf.coef_[idx]
    contribs = [(names[j], float(v * coefs[j])) for j, v in zip(row.col, row.data)]
    contribs.sort(key=lambda kv: -abs(kv[1]))
    return classes, probs, classes[idx], contribs[:top_k]
'''

(ROOT / "src").mkdir(exist_ok=True)
(ROOT / "src" / "__init__.py").write_text("")
(ROOT / "src" / "text_utils.py").write_text(TEXT_UTILS_SOURCE.strip("\n") + "\n")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))
print("Wrote", ROOT / "src" / "text_utils.py")

In [ ]:
from src.text_utils import clean_text, light_clean, explain_linear

examples = ["I can't believe how good this is!!! sooooo tasty, will buy again http://example.com/x",
            "Don't buy this. The package arrived damaged and it was NOT what the picture showed :("]
for e in examples:
    print("original   :", e)
    print("clean_text :", clean_text(e))
    print("no stopword:", clean_text(e, remove_stopwords=True))
    print("light_clean:", light_clean(e), "\n")

## 4. Train / validation / test split

70 % train, 15 % validation (model selection), 15 % test (final, one-time evaluation), **stratified** so every part keeps the same class proportions. All vectorizers are fitted on the training data only, so nothing leaks from validation or test.

In [ ]:
PREP = {                                           # text column name -> preprocessing function
    "clean": clean_text,
    "clean_nosw": lambda t: clean_text(t, remove_stopwords=True),
    "light": light_clean,
}

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["sentiment"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["sentiment"], random_state=SEED)

for d in (train_df, val_df, test_df):
    for col, fn in PREP.items():
        d[col] = d["text"].apply(fn)

print("train:", train_df.shape, " val:", val_df.shape, " test:", test_df.shape)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:>5} class share:", d["sentiment"].value_counts(normalize=True).round(3).to_dict())

## 5. Evaluation helpers and baselines

In [ ]:
results = []        # one row per model: validation scores
predictors = {}     # model name -> (predict function returning label strings, text column it expects)

def register(name, predict_fn, col="clean", note=""):
    # Score a model on the validation set; col = text column the model was trained on
    pred = predict_fn(val_df[col])
    results.append({
        "model": name,
        "val_accuracy": accuracy_score(val_df["sentiment"], pred),
        "val_macro_f1": f1_score(val_df["sentiment"], pred, average="macro"),
        "note": note,
    })
    predictors[name] = (predict_fn, col)

def plot_confusion(ax, y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=LABELS, normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
    ax.set_title(title + "\n(rows = true class)")

### 5.1 Majority-class "model"
Always predicts the most common class. Every real model must beat this.

In [ ]:
majority = train_df["sentiment"].mode()[0]
register("Majority class", lambda texts: np.full(len(texts), majority), note="always predicts the most common class")
pd.DataFrame(results).round(4)

### 5.2 TF-IDF + Naive Bayes and Logistic Regression
TF-IDF turns each review into a weighted word-count vector (unigrams and bigrams, so *not good* is visible as a pair). These models are fast, strong on short text and interpretable, which makes them the right baseline.

In [ ]:
def make_tfidf(max_features=50000, ngram=(1, 2)):
    return TfidfVectorizer(ngram_range=ngram, min_df=2, max_features=max_features, sublinear_tf=True)

lr_base = Pipeline([("tfidf", make_tfidf()), ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])
lr_base.fit(train_df["clean"], train_df["sentiment"])
register("TF-IDF + LogReg (baseline)", lr_base.predict)

nb_base = Pipeline([("tfidf", make_tfidf()), ("clf", MultinomialNB(alpha=0.5))])
nb_base.fit(train_df["clean"], train_df["sentiment"])
register("TF-IDF + Naive Bayes", nb_base.predict)

pd.DataFrame(results).round(4)

## 6. Ablation: does stopword removal help?

Same model, two versions of the text. A preprocessing step is kept only if the validation score supports it.

In [ ]:
ablation = []
for col, label in [("clean", "keep stopwords"), ("clean_nosw", "remove stopwords")]:
    pipe = Pipeline([("tfidf", make_tfidf()), ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])
    pipe.fit(train_df[col], train_df["sentiment"])
    pred = pipe.predict(val_df[col])
    ablation.append({"preprocessing": label,
                     "val_accuracy": accuracy_score(val_df["sentiment"], pred),
                     "val_macro_f1": f1_score(val_df["sentiment"], pred, average="macro")})
ablation = pd.DataFrame(ablation).round(4)
display(ablation)

best_col = "clean" if ablation.loc[0, "val_macro_f1"] >= ablation.loc[1, "val_macro_f1"] else "clean_nosw"
print(f"-> Using the '{best_col}' text for the tuned linear model.")

## 7. Tuned Logistic Regression

5-fold cross-validated grid search on the training set, optimising macro-F1. This is the model that will be shipped in the app as the fast, interpretable option.

In [ ]:
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [20000, 50000],
    "clf__C": [0.3, 1, 3, 10],
}
pipe = Pipeline([("tfidf", TfidfVectorizer(min_df=2, sublinear_tf=True)),
                 ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])

grid = GridSearchCV(pipe, param_grid, scoring="f1_macro",
                    cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=-1)
grid.fit(train_df[best_col], train_df["sentiment"])

print("Best params:", grid.best_params_)
print("Best CV macro-F1: %.4f" % grid.best_score_)
lr_tuned = grid.best_estimator_

LR_NAME = "TF-IDF + LogReg (tuned)"
register(LR_NAME, lr_tuned.predict, col=best_col, note=str(grid.best_params_))
pd.DataFrame(results).round(4)

## 8. Fine-tuning a transformer

A pretrained transformer already "knows" English, so fine-tuning it on a few thousand labelled reviews (about 21k with the default sample) teaches it the task. Choices:

- **RoBERTa-base**: a stronger BERT-style model (125M parameters) than DistilBERT, at about twice the cost. Swap in `distilroberta-base` for speed.
- Minimal text cleaning (`light_clean`), 3 epochs, linear learning-rate schedule with warm-up, mixed precision on GPU.
- The best epoch by **validation macro-F1** is kept (same early-stopping idea as before).
- A plain PyTorch loop is used instead of a high-level trainer, so every step is visible and independent of library version changes.

If PyTorch/transformers are missing or `RUN_TRANSFORMER = False`, this section is skipped and the rest of the notebook still works.

In [ ]:
try:
    import torch
    from torch.utils.data import Dataset, DataLoader
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              DataCollatorWithPadding, get_linear_schedule_with_warmup)
    TORCH_OK = True
except ImportError as e:
    TORCH_OK = False
    print("PyTorch / transformers not available, skipping the transformer:", e)

USE_TRANSFORMER = RUN_TRANSFORMER and TORCH_OK
transformer_ready = False
TR_NAME = TRANSFORMER_NAME.split("/")[-1] + " (fine-tuned)"

if USE_TRANSFORMER:
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    if device.type == "cpu" and TRANSFORMER_TRAIN_SIZE is None:
        print("WARNING: no GPU detected. Training on the full data will be slow; "
              "set TRANSFORMER_TRAIN_SIZE (e.g. 4000) or enable a GPU.")

    label2id = {l: i for i, l in enumerate(LABELS)}
    id2label = {i: l for l, i in label2id.items()}
    tok = AutoTokenizer.from_pretrained(TRANSFORMER_NAME)
    collator = DataCollatorWithPadding(tok)

    class ReviewDataset(Dataset):
        def __init__(self, texts, labels=None):
            self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN)
            self.labels = None if labels is None else [label2id[l] for l in labels]
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            item = {k: v[i] for k, v in self.enc.items()}
            if self.labels is not None:
                item["labels"] = self.labels[i]
            return item

    @torch.no_grad()
    def transformer_proba(texts, batch_size=64):
        model.eval()
        loader = DataLoader(ReviewDataset(texts), batch_size=batch_size, collate_fn=collator)
        out = []
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out.append(torch.softmax(model(**batch).logits.float(), dim=-1).cpu().numpy())
        return np.vstack(out)

    def transformer_predict(texts):
        return np.array(LABELS)[transformer_proba(texts).argmax(axis=1)]

In [ ]:
if USE_TRANSFORMER:
    tr = train_df if TRANSFORMER_TRAIN_SIZE is None else train_df.sample(TRANSFORMER_TRAIN_SIZE, random_state=SEED)
    print("Training reviews:", len(tr))

    model = AutoModelForSequenceClassification.from_pretrained(
        TRANSFORMER_NAME, num_labels=3, id2label=id2label, label2id=label2id).to(device)

    # Multi-GPU: `model` stays the plain model (used for evaluation, saving and restoring the best weights).
    # `train_model` is the same model wrapped in DataParallel only when several GPUs are used; each batch is then
    # split across the GPUs (batch 32 on 2 GPUs = 16 per GPU), so the hyperparameters and results stay comparable.
    N_GPUS = torch.cuda.device_count() if device.type == "cuda" else 0
    GPUS_USED = N_GPUS if (USE_ALL_GPUS and N_GPUS > 1) else min(N_GPUS, 1)
    train_model = torch.nn.DataParallel(model) if GPUS_USED > 1 else model
    print(f"GPUs used for training: {GPUS_USED}" + (f" (each batch of {TRANSFORMER_BATCH} is split across them)" if GPUS_USED > 1 else ""))

    train_loader = DataLoader(ReviewDataset(tr["light"], tr["sentiment"]),
                              batch_size=TRANSFORMER_BATCH, shuffle=True, collate_fn=collator)
    total_steps = len(train_loader) * TRANSFORMER_EPOCHS
    optimizer = torch.optim.AdamW(model.parameters(), lr=TRANSFORMER_LR, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * total_steps), total_steps)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    history, best_f1, best_state = [], -1.0, None
    t0 = time.time()
    for epoch in range(1, TRANSFORMER_EPOCHS + 1):
        model.train()
        running = 0.0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = train_model(**batch).loss.mean()   # .mean(): DataParallel returns one loss per GPU
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item()

        val_pred = transformer_predict(val_df["light"])
        f1 = f1_score(val_df["sentiment"], val_pred, average="macro")
        acc = accuracy_score(val_df["sentiment"], val_pred)
        history.append({"epoch": epoch, "train_loss": running / len(train_loader),
                        "val_accuracy": acc, "val_macro_f1": f1})
        print(f"epoch {epoch}: train loss {running / len(train_loader):.4f} | val acc {acc:.4f} | val macro-F1 {f1:.4f}")
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    train_minutes = (time.time() - t0) / 60
    print(f"Done in {train_minutes:.1f} min. Best validation macro-F1: {best_f1:.4f}")

    register(TR_NAME, transformer_predict, col="light",
             note=f"{TRANSFORMER_NAME}, {TRANSFORMER_EPOCHS} epochs, max_len {MAX_LEN}")
    transformer_ready = True

In [ ]:
if transformer_ready:
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    axes[0].plot(h["epoch"], h["train_loss"], marker="o"); axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch")
    axes[1].plot(h["epoch"], h["val_macro_f1"], marker="o", label="macro-F1")
    axes[1].plot(h["epoch"], h["val_accuracy"], marker="o", label="accuracy")
    axes[1].set_title("Validation score"); axes[1].set_xlabel("epoch"); axes[1].legend()
    plt.tight_layout()
    save_fig(fig, "transformer_training.png")
    plt.show()
else:
    print("Transformer not trained in this run.")

## 9. Model comparison (validation set), including prediction speed

Accuracy is only one criterion. A model that is 1 % better but 50x slower may be the wrong choice for a high-volume service, so the speed per review is measured too (text preprocessing excluded; hardware is whatever this notebook runs on).

In [ ]:
def ms_per_tweet(name, n=500):
    fn, col = predictors[name]
    sample = test_df[col].iloc[:n]
    fn(sample.iloc[:16])                      # warm-up
    t = time.perf_counter()
    fn(sample)
    return (time.perf_counter() - t) / len(sample) * 1000

shipped = [LR_NAME] + ([TR_NAME] if transformer_ready else [])
latency = {name: ms_per_tweet(name) for name in shipped}

res = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
res["ms_per_tweet"] = res["model"].map(latency)
display(res.round(4))

fig, ax = plt.subplots(figsize=(8, 3.5))
sns.barplot(data=res, y="model", x="val_macro_f1", ax=ax)
ax.set_xlim(0, 1); ax.set_title("Validation macro-F1 by model")
save_fig(fig, "model_comparison.png")
plt.show()

## 10. Final evaluation on the test set

The test set has not influenced any decision so far. Both shipped models are scored **once** here.

In [ ]:
test_metrics = {}
fig, axes = plt.subplots(1, len(shipped), figsize=(5 * len(shipped), 4.5), squeeze=False)
for ax, name in zip(axes[0], shipped):
    fn, col = predictors[name]
    y_pred = fn(test_df[col])
    y_true = test_df["sentiment"]
    rep = classification_report(y_true, y_pred, labels=LABELS, digits=3, output_dict=True)
    test_metrics[name] = {"accuracy": float(accuracy_score(y_true, y_pred)),
                          "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
                          "per_class": {l: {k: float(rep[l][k]) for k in ("precision", "recall", "f1-score")} for l in LABELS}}
    print(f"===== {name} =====")
    print(f"accuracy {test_metrics[name]['accuracy']:.4f} | macro-F1 {test_metrics[name]['macro_f1']:.4f}")
    print(classification_report(y_true, y_pred, labels=LABELS, digits=3))
    plot_confusion(ax, y_true, y_pred, name)
plt.tight_layout()
save_fig(fig, "confusion_matrices.png")
plt.show()

## 10b. Natural-distribution check

The test set above is balanced, which is the right choice for comparing models fairly, but real product reviews are mostly positive. Here both models are scored on a random holdout of reviews that was set aside **before** the capped sample was drawn, so it keeps the natural class mix. Expect accuracy to look high because the positive class dominates, so read macro-F1 and the per-class **recall** (how many real negative and neutral reviews were found) rather than accuracy.

In [ ]:
nat_metrics = {}
rows = []
print("Class share of the natural check set:", natural_df["sentiment"].value_counts(normalize=True).reindex(LABELS).fillna(0).round(3).to_dict())
print(f"Always predicting the most common class would score {natural_df['sentiment'].value_counts(normalize=True).max():.1%} accuracy\n")
for name in shipped:
    fn, col = predictors[name]
    pred = fn(natural_df["text"].apply(PREP[col]))
    rep_ = classification_report(natural_df["sentiment"], pred, labels=LABELS, output_dict=True, zero_division=0)
    nat_metrics[name] = {"accuracy": float(accuracy_score(natural_df["sentiment"], pred)),
                         "macro_f1": float(f1_score(natural_df["sentiment"], pred, average="macro")),
                         "recall": {l: float(rep_[l]["recall"]) for l in LABELS},
                         "precision": {l: float(rep_[l]["precision"]) for l in LABELS}}
    rows.append({"model": name, "accuracy": nat_metrics[name]["accuracy"], "macro-F1": nat_metrics[name]["macro_f1"],
                 **{f"recall {l}": nat_metrics[name]["recall"][l] for l in LABELS},
                 **{f"precision {l}": nat_metrics[name]["precision"][l] for l in LABELS}})
display(pd.DataFrame(rows).round(3))

## 11. Cross-domain generalisation (reviews to tweets)

Scores on held-out reviews of the same dataset can be optimistic. To measure how far the models transfer, both are evaluated **without retraining** on a different kind of text: **TweetEval** sentiment tweets (same three labels). A large drop is expected because tweets are short, informal and labelled differently from star ratings; how large it is, and for which model, is the interesting finding.

If the dataset cannot be downloaded (no internet, e.g. on Kaggle), this section is skipped. Alternatively set `CROSS_DATASET_CSV` to a local CSV with `text` and `sentiment` columns (values `negative`, `neutral`, `positive`).

In [ ]:
def load_cross_dataset(n):
    if CROSS_DATASET_CSV:
        other = pd.read_csv(CROSS_DATASET_CSV)[["text", "sentiment"]].dropna()
        source = CROSS_DATASET_CSV
    else:
        from datasets import load_dataset
        last_err = None
        for name in ["tweet_eval", "cardiffnlp/tweet_eval"]:
            try:
                ds = load_dataset(name, "sentiment", split="test")
                break
            except Exception as e:      # noqa
                last_err = e
        else:
            raise last_err
        names = ds.features["label"].names
        other = pd.DataFrame({"text": ds["text"], "sentiment": [names[i] for i in ds["label"]]})
        source = "TweetEval (sentiment, test split)"
    other = other[other["sentiment"].isin(LABELS)]
    if len(other) > n:
        other = other.sample(n, random_state=SEED)
    return other.reset_index(drop=True), source

cross_metrics, cross_source = {}, None
try:
    other, cross_source = load_cross_dataset(CROSS_DATASET_N)
    print(f"Cross-domain: {cross_source}, {len(other)} texts")
    print(other["sentiment"].value_counts(normalize=True).round(3).to_dict())

    rows = []
    for name in shipped:
        fn, col = predictors[name]
        pred = fn(other["text"].apply(PREP[col]))
        acc = accuracy_score(other["sentiment"], pred)
        f1 = f1_score(other["sentiment"], pred, average="macro")
        cross_metrics[name] = {"accuracy": float(acc), "macro_f1": float(f1)}
        rows.append({"model": name,
                     "in-domain macro-F1": test_metrics[name]["macro_f1"], "other-dataset macro-F1": f1,
                     "drop": test_metrics[name]["macro_f1"] - f1})
    cross_table = pd.DataFrame(rows)
    display(cross_table.round(4))
except Exception as e:                  # noqa
    print("Cross-dataset evaluation skipped:", type(e).__name__, "-", str(e)[:200])

## 12. Interpretability: what did the models learn?

For the linear model the coefficients are directly readable. The same logic powers the per-review explanation in the Streamlit app: each word's contribution is its TF-IDF weight times the coefficient for the predicted class.

In [ ]:
vec = lr_tuned.named_steps["tfidf"]
clf = lr_tuned.named_steps["clf"]
names = np.array(vec.get_feature_names_out())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (i, cls) in zip(axes, enumerate(clf.classes_)):
    pd.Series(clf.coef_[i], index=names).nlargest(12).sort_values().plot.barh(ax=ax)
    ax.set_title(f"Words pushing towards: {cls}")
plt.tight_layout()
save_fig(fig, "lr_top_words.png")
plt.show()

In [ ]:
demo_reviews = ["Absolutely delicious, best coffee I have ever bought. Will order again!",
                "Terrible. Arrived stale and the taste was awful. Not worth the money.",
                "It is okay. Nothing special but not bad either.",
                "not bad at all"]
remove_sw = (best_col == "clean_nosw")
for t in demo_reviews:
    classes, probs, pred, contribs = explain_linear(lr_tuned, clean_text(t, remove_stopwords=remove_sw), top_k=5)
    print(f"{t!r}\n   -> {pred} ({probs.max():.0%}); main terms: " + ", ".join(f"{w} ({s:+.2f})" for w, s in contribs) + "\n")

## 13. Error analysis

In [ ]:
primary = max(shipped, key=lambda n: res.set_index("model").loc[n, "val_macro_f1"])
fn, col = predictors[primary]
errors = test_df.assign(pred=fn(test_df[col]))
errors = errors[errors["sentiment"] != errors["pred"]]
print(f"Primary model: {primary}")
print(f"{len(errors)} of {len(test_df)} test reviews misclassified ({len(errors) / len(test_df):.1%})\n")

print("Most common mistakes (true -> predicted):")
print(errors.groupby(["sentiment", "pred"]).size().sort_values(ascending=False).head(5).to_string(), "\n")
errors[["text", "sentiment", "pred"]].sample(min(12, len(errors)), random_state=SEED).rename(columns={"sentiment": "true"})

When reading these examples, look for recurring patterns: **sarcasm**, very **short reviews** with no sentiment words, **mixed feelings** in one review, and **3-star** reviews that contain strong words. The labels themselves are subjective (neutral versus mildly positive/negative), which caps the achievable score and explains part of the confusion between neighbouring classes.

## 14. Save the models for the Streamlit app

This writes everything `app.py` needs into `models/` and the figures into `assets/`:

- `models/lr_pipeline.joblib`: the tuned TF-IDF + Logistic Regression pipeline
- `models/transformer/`: the fine-tuned transformer and tokenizer (if trained)
- `models/metadata.json`: labels, preprocessing choice, metrics, speed and library versions (the app displays this)
- `src/text_utils.py` (already written above): shared preprocessing

In [ ]:
joblib.dump(lr_tuned, MODELS_DIR / "lr_pipeline.joblib", compress=3)

if transformer_ready:
    model.save_pretrained(MODELS_DIR / "transformer")
    tok.save_pretrained(MODELS_DIR / "transformer")

def split_summary(d):
    return {"rows": int(len(d)), "class_share": {k: round(float(v), 4) for k, v in d["sentiment"].value_counts(normalize=True).items()}}

val_scores = res.set_index("model")[["val_accuracy", "val_macro_f1"]]
versions = {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
            "pandas": pd.__version__, "numpy": np.__version__}
if TORCH_OK:
    import transformers
    versions.update({"torch": torch.__version__, "transformers": transformers.__version__})

metadata = {
    "created": date.today().isoformat(),
    "task": "3-class product-review sentiment from star ratings (1-2 negative, 3 neutral, 4-5 positive)",
    "domain": "reviews", "item_name": "review",
    "page_title": "Review Sentiment Classifier", "app_title": "🛒 Review Sentiment",
    "caption": "Classifies English product reviews as negative, neutral or positive. Trained on star-rated Amazon-style reviews; it can misread sarcasm, and 3-star reviews are often mixed.",
    "examples": ["Absolutely delicious, best coffee I have ever bought. Will order again!",
                 "Terrible. Arrived stale and the taste was awful. Not worth the money.",
                 "It is okay. Nothing special but not bad either.",
                 "Great price, but the packaging was damaged and shipping took two weeks.",
                 "Does exactly what it says. Good value."],
    "labels": LABELS,
    "dataset": {"name": f"Product reviews ({Path(str(data_path)).name}), capped at {PER_CLASS_CAP} per class", "total_rows_after_cleaning": int(len(df)),
                "unique_reviews_before_sampling": int(len(df_all)), "natural_class_share": {k: round(float(v), 4) for k, v in df_all["sentiment"].value_counts(normalize=True).items()},
                "train": split_summary(train_df), "validation": split_summary(val_df), "test": split_summary(test_df)},
    "lr": {"file": "lr_pipeline.joblib", "display_name": LR_NAME, "remove_stopwords": bool(best_col == "clean_nosw"),
           "best_params": {k: str(v) for k, v in grid.best_params_.items()}, "cv_macro_f1": float(grid.best_score_),
           "val_accuracy": float(val_scores.loc[LR_NAME, "val_accuracy"]), "val_macro_f1": float(val_scores.loc[LR_NAME, "val_macro_f1"]),
           "test": test_metrics[LR_NAME], "ms_per_tweet": float(latency[LR_NAME])},
    "transformer": None,
    "cross_dataset": {"source": cross_source, "n": CROSS_DATASET_N, "metrics": cross_metrics} if cross_metrics else None,
    "natural_check": {"n": int(len(natural_df)), "metrics": nat_metrics},
    "versions": versions,
}
if transformer_ready:
    metadata["transformer"] = {"dir": "transformer", "display_name": TR_NAME, "base_model": TRANSFORMER_NAME,
                               "epochs": TRANSFORMER_EPOCHS, "max_len": MAX_LEN, "train_tweets": int(len(tr)),
                               "train_minutes": float(train_minutes), "device": f"{device} (GPUs used: {GPUS_USED})",
                               "val_accuracy": float(val_scores.loc[TR_NAME, "val_accuracy"]),
                               "val_macro_f1": float(val_scores.loc[TR_NAME, "val_macro_f1"]),
                               "test": test_metrics[TR_NAME], "ms_per_tweet": float(latency[TR_NAME])}

with open(MODELS_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

for p in sorted(MODELS_DIR.rglob("*")):
    if p.is_file():
        print(f"{str(p):<45} {p.stat().st_size / 1e6:8.2f} MB")

**Sanity check:** reload the saved files from disk the way the app will, and confirm the predictions match.

In [ ]:
check_tweets = ["Absolutely delicious, best coffee I have ever bought!", "Terrible, arrived stale and awful taste", "It is okay, nothing special"]

lr_loaded = joblib.load(MODELS_DIR / "lr_pipeline.joblib")
prep = lambda t: clean_text(t, remove_stopwords=metadata["lr"]["remove_stopwords"])
assert list(lr_loaded.predict([prep(t) for t in check_tweets])) == list(lr_tuned.predict([prep(t) for t in check_tweets]))
print("LogReg reload OK:", list(lr_loaded.predict([prep(t) for t in check_tweets])))

if transformer_ready:
    tok2 = AutoTokenizer.from_pretrained(MODELS_DIR / "transformer")
    model2 = AutoModelForSequenceClassification.from_pretrained(MODELS_DIR / "transformer").to(device).eval()
    enc = tok2([light_clean(t) for t in check_tweets], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        reloaded = [model2.config.id2label[i] for i in model2(**enc).logits.argmax(-1).tolist()]
    original = list(transformer_predict([light_clean(t) for t in check_tweets]))
    assert reloaded == original, (reloaded, original)
    print("Transformer reload OK:", reloaded)

## 15. Results summary

In [ ]:
rows = []
for name in shipped:
    t = test_metrics[name]
    row = {"Model": name, "Test accuracy": f"{t['accuracy']:.3f}", "Test macro-F1": f"{t['macro_f1']:.3f}"}
    row["Natural-mix macro-F1"] = f"{nat_metrics[name]['macro_f1']:.3f}"
    if name in cross_metrics:
        row["Other-dataset macro-F1"] = f"{cross_metrics[name]['macro_f1']:.3f}"
    row["ms / review"] = f"{latency[name]:.2f}"
    rows.append(row)
summary = pd.DataFrame(rows)
display(summary)

# README-ready markdown table
try:
    table_md = summary.to_markdown(index=False)
except ImportError:                      # pandas needs 'tabulate' for to_markdown
    cols = list(summary.columns)
    table_md = "\n".join(["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)] +
                         ["| " + " | ".join(map(str, r)) + " |" for r in summary.values])
(ASSETS_DIR / "results.md").write_text(table_md + "\n")
print("\nSaved README-ready table to", ASSETS_DIR / "results.md")

In [ ]:
# Findings generated from the numbers above (so they are always consistent with the results)
lr_t = test_metrics[LR_NAME]
print(f"* Majority-class baseline reaches {res.set_index('model').loc['Majority class', 'val_accuracy']:.1%} validation accuracy "
      f"and macro-F1 {res.set_index('model').loc['Majority class', 'val_macro_f1']:.2f} on the class-capped sample (about chance for 3 classes).")
print(f"* Stopword removal: {'helped' if best_col == 'clean_nosw' else 'did not help'} "
      f"(validation macro-F1 {ablation.loc[0, 'val_macro_f1']:.3f} keeping vs {ablation.loc[1, 'val_macro_f1']:.3f} removing).")
print(f"* Tuned Logistic Regression: test macro-F1 {lr_t['macro_f1']:.3f}, {latency[LR_NAME]:.2f} ms per review.")
if transformer_ready:
    tr_t = test_metrics[TR_NAME]
    gain = tr_t["macro_f1"] - lr_t["macro_f1"]
    print(f"* {TR_NAME}: test macro-F1 {tr_t['macro_f1']:.3f} ({gain:+.3f} vs Logistic Regression), "
          f"but about {latency[TR_NAME] / max(latency[LR_NAME], 1e-9):.0f}x slower per review on this hardware.")
weakest = {n: min(test_metrics[n]["per_class"], key=lambda c: test_metrics[n]["per_class"][c]["recall"]) for n in shipped}
for n, c in weakest.items():
    print(f"* {n}: lowest recall is on '{c}' ({test_metrics[n]['per_class'][c]['recall']:.2f}).")
for n in shipped:
    nm = nat_metrics[n]
    print(f"* Natural class mix: {n} reaches accuracy {nm['accuracy']:.3f}, macro-F1 {nm['macro_f1']:.3f}; "
          f"recall negative {nm['recall']['negative']:.2f}, neutral {nm['recall']['neutral']:.2f}, positive {nm['recall']['positive']:.2f}.")
if cross_metrics:
    for n in shipped:
        print(f"* On tweets (a different domain), {n} scores macro-F1 {cross_metrics[n]['macro_f1']:.3f} "
              f"(in-domain {test_metrics[n]['macro_f1']:.3f}).")

**Next:** if you trained on Kaggle or Colab, download the `models/` and `assets/` folders into the project folder (next to `app.py`), then run `streamlit run app.py` from the project root. See the README.